# Support Ticket Data Cleaning

This notebook turns the messy raw export (`data/support_tickets_raw.csv`) into the clean, analysis-ready dataset used everywhere else in this project (`data/support_tickets_cleaned.csv` — the file `sql/01_schema_and_load.sql` loads into MySQL).

**Data quality issues found in the raw export, and how each is handled:**

| Issue | Rows affected | Fix |
|---|---|---|
| Duplicate `ticket_id` (ticket exported twice) | ~100 | Drop duplicates, keep first occurrence |
| Inconsistent `category` text (case, whitespace, typos) | ~700 | Normalize to 6 canonical category names |
| Impossible negative `resolution_time_hours` | 10 | Take absolute value (sign is a clear data-entry error, not a real negative duration) |
| Missing `agent` (ticket never assigned) | ~50 | Fill with an explicit `'Unknown'` placeholder — kept visible rather than silently dropped, and excluded from per-agent KPIs in `sql/02_kpi_queries.sql` |
| Missing `satisfaction_score` (customer never responded to the CSAT survey) | ~1,800 | Left as a true missing value — this is a real non-response, not a data error, so it is **not** imputed. `AVG()` and similar aggregates skip missing values on their own. |

Run this notebook top to bottom (Kernel → Restart & Run All) to reproduce `data/support_tickets_cleaned.csv` from scratch.

In [1]:
import re
import pandas as pd

raw = pd.read_csv("../data/support_tickets_raw.csv")
print(f"Raw rows: {len(raw)}")
raw.head()

Raw rows: 5100


,ticket_id,created_date,resolved_date,channel,category,priority,agent,resolution_time_hours,satisfaction_score,status
0,TCK-01909,2025-01-01,2025-01-01 02:29:24.000000000,Email,Product Question,Medium,Agent_06,2.49,5.0,Closed
1,TCK-01448,2025-01-01,2025-01-01 03:08:24.000000000,Email,Technical Issue,Medium,Agent_10,3.14,4.0,Resolved
2,TCK-01369,2025-01-01,2025-01-01 10:21:00.000000000,Chat,Product Question,Medium,Agent_15,10.35,3.0,Resolved
3,TCK-01956,2025-01-01,2025-01-01 06:17:23.999999999,Phone,Refund Request,High,Agent_13,6.29,NaN,Resolved
4,TCK-01442,2025-01-01,2025-01-01 13:52:48.000000000,Email,Technical Issue,Low,Agent_09,13.88,NaN,Resolved


## Step 1: Understand what's wrong with the raw data

Before fixing anything, measure the damage.

In [2]:
print("=== Data quality report: raw export ===")
print(f"Duplicate ticket_id rows:      {raw['ticket_id'].duplicated().sum()}")
print(f"Distinct category strings:     {raw['category'].nunique()} (should be 6)")
print(sorted(raw['category'].unique()))
print(f"Negative resolution_time_hours (impossible): {(raw['resolution_time_hours'] < 0).sum()}")
print(f"Missing agent:                 {raw['agent'].isna().sum()}")
print(f"Missing satisfaction_score:    {raw['satisfaction_score'].isna().sum()} ({raw['satisfaction_score'].isna().mean():.1%})")

=== Data quality report: raw export ===
Duplicate ticket_id rows:      100
Distinct category strings:     20 (should be 6)
[' Billing', 'Account  Access', 'Account Access', 'Bill', 'Billing', 'COMPLAINT', 'Complaint', 'Product Question', 'Product Qustion', 'Refund Reqeust', 'Refund Request', 'TECH ISSUE', 'Technical Issue', 'Technical issue ', 'account access', 'billing', 'complaint', 'product question', 'refund request', 'technical issue']
Negative resolution_time_hours (impossible): 10
Missing agent:                 54
Missing satisfaction_score:    1845 (36.2%)


## Step 2: Remove duplicate ticket exports

Same `ticket_id` appearing twice means the same ticket got exported twice, not two different tickets. Keep the first occurrence, drop the rest.

In [3]:
df = raw.drop_duplicates(subset="ticket_id", keep="first").reset_index(drop=True)
print(f"Rows after dedup: {len(df)} (removed {len(raw) - len(df)})")

Rows after dedup: 5000 (removed 100)


## Step 3: Standardize category text

The raw export has the same 6 categories typed inconsistently: different casing (`TECH ISSUE` vs `technical issue`), stray whitespace (`Account  Access`), abbreviations (`Bill`), and outright typos (`Product Qustion`, `Refund Reqeust`). A lookup table maps every known variant back to its canonical form.

In [4]:
CATEGORY_MAP = {
    "billing": "Billing",
    "bill": "Billing",
    "technical issue": "Technical Issue",
    "tech issue": "Technical Issue",
    "account access": "Account Access",
    "product question": "Product Question",
    "product qustion": "Product Question",   # typo in source export
    "complaint": "Complaint",
    "refund request": "Refund Request",
    "refund reqeust": "Refund Request",      # typo in source export
}

def clean_category(value):
    normalized = re.sub(r"\s+", " ", str(value).strip().lower())
    return CATEGORY_MAP.get(normalized, value)

df["category"] = df["category"].apply(clean_category)

print(f"Distinct categories after cleaning: {df['category'].nunique()}")
print(sorted(df["category"].unique()))

Distinct categories after cleaning: 6
['Account Access', 'Billing', 'Complaint', 'Product Question', 'Refund Request', 'Technical Issue']


## Step 4: Fix impossible negative resolution times

A resolution time can't be negative. These 10 rows are a sign-flip data-entry error, not a real measurement, so the fix is to take the absolute value rather than drop the rows.

In [5]:
n_negative = (df["resolution_time_hours"] < 0).sum()
df["resolution_time_hours"] = df["resolution_time_hours"].abs()
print(f"Fixed {n_negative} rows with a negative resolution_time_hours")

Fixed 10 rows with a negative resolution_time_hours


## Step 5: Handle missing agent

A ticket with no assigned agent (auto-assigned then unassigned) still needs to exist in the dataset for volume/category KPIs — it just isn't attributable to any one agent's performance. Fill with an explicit `'Unknown'` label instead of leaving it blank, so it's visible rather than silently disappearing.

In [6]:
n_missing_agent = df["agent"].isna().sum()
df["agent"] = df["agent"].fillna("Unknown")
print(f"Filled {n_missing_agent} missing agent values with 'Unknown'")
print("Note: sql/02_kpi_queries.sql explicitly excludes 'Unknown' from the per-agent KPI - it isn't a real agent.")

Filled 50 missing agent values with 'Unknown'
Note: sql/02_kpi_queries.sql explicitly excludes 'Unknown' from the per-agent KPI - it isn't a real agent.


## Step 6: Missing satisfaction_score - left as-is, on purpose

~35% of tickets have no CSAT score. That matches real survey response rates (most customers never fill them out) - it's not a data error to fix. Filling it with a guessed value (e.g. the average) would fabricate opinions customers never gave and quietly bias any satisfaction KPI. It's left as a missing value; `AVG()` and similar functions already skip it correctly.

## Step 7: Final validation

Check the cleaning actually worked before saving anything.

In [7]:
assert len(df) == 5000, "expected 5000 unique tickets after dedup"
assert df["category"].nunique() == 6, "expected exactly 6 canonical categories"
assert (df["resolution_time_hours"] < 0).sum() == 0, "no negative durations should remain"
assert df["agent"].isna().sum() == 0, "no missing agent values should remain"
print("All validation checks passed.")
df.describe(include="all")

All validation checks passed.


,ticket_id,created_date,resolved_date,channel,category,priority,agent,resolution_time_hours,satisfaction_score,status
count,5000,5000,5000,5000,5000,5000,5000,5000.000000,3192.000000,5000
unique,5000,365,4958,4,6,4,21,NaN,NaN,3
top,TCK-01909,2025-11-07,2025-10-20 03:27:36.000000000,Email,Technical Issue,Medium,Agent_05,NaN,NaN,Resolved
freq,1,27,3,2286,1468,1963,269,NaN,NaN,3790
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.827778,3.763471,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.659996,0.883786,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.190000,1.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.380000,3.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.860000,4.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.160000,4.000000,NaN


## Step 8: Save the cleaned dataset

In [8]:
df.to_csv("../data/support_tickets_cleaned.csv", index=False)
print(f"Saved {len(df)} rows to data/support_tickets_cleaned.csv")

Saved 5000 rows to data/support_tickets_cleaned.csv
